# Multi-Objective Optimization Analysis Notebook

This notebook provides comprehensive analysis of the multi-objective optimization results for building footprint segmentation.

## Contents
1. Setup and Data Loading
2. Pareto Front Analysis
3. Model Comparison
4. Training Curves
5. Qualitative Results

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add parent directory to path
sys.path.append('..')

from src.pareto import extract_pareto_front, plot_pareto_front
from src.evaluate import generate_comparison_table
from visualization.visualization import plot_training_history, plot_prediction_grid

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

2026-04-14 22:59:44.907334: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776194984.931216  174033 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776194984.936548  174033 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776194984.953729  174033 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776194984.953752  174033 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776194984.953754  174033 computation_placer.cc:177] computation placer alr

ImportError: cannot import name 'save_sample_predictions' from 'visualization.visualization' (/home/reza/Dropbox/Research/Collaborations/Mr Khoshnevisan/Multi-objective Semantic Segmentation/Codes/01/notebooks/../visualization/visualization.py)

## 1. Load Results

In [ ]:
# Path to results directory
RESULTS_DIR = '../outputs'

# Find the most recent run
runs = sorted([d for d in os.listdir(RESULTS_DIR) if os.path.isdir(os.path.join(RESULTS_DIR, d))])
latest_run = runs[-1] if runs else None

if latest_run:
    run_path = os.path.join(RESULTS_DIR, latest_run)
    print(f"Loading results from: {run_path}")
    
    # Load results CSV
    results_path = os.path.join(run_path, 'tables', 'results.csv')
    if os.path.exists(results_path):
        df = pd.read_csv(results_path)
        print(f"Loaded {len(df)} experiments")
        display(df.head())
    else:
        print("No results.csv found")
else:
    print("No runs found")

## 2. Pareto Front Analysis

In [ ]:
# Extract Pareto front
if 'df' in locals():
    results = df.to_dict('records')
    pareto_results, pareto_values = extract_pareto_front(results)
    
    print(f"Found {len(pareto_results)} Pareto-optimal solutions out of {len(results)} total")
    
    # Display Pareto front table
    pareto_df = pd.DataFrame(pareto_results)
    display(pareto_df[['architecture', 'encoder', 'iou', 'f1_score', 'param_count', 'inference_time']].head(10))

In [ ]:
# Plot Pareto fronts
if 'df' in locals():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # IoU vs Parameters
    ax = axes[0, 0]
    ax.scatter(df['param_count'] / 1e6, df['iou'], alpha=0.5, label='All')
    if len(pareto_results) > 0:
        ax.scatter([r['param_count'] / 1e6 for r in pareto_results], 
                  [r['iou'] for r in pareto_results], 
                  c='red', s=100, marker='*', label='Pareto-optimal')
    ax.set_xlabel('Parameters (M)')
    ax.set_ylabel('IoU')
    ax.set_title('IoU vs Model Size')
    ax.legend()
    
    # IoU vs Inference Time
    ax = axes[0, 1]
    ax.scatter(df['inference_time'], df['iou'], alpha=0.5, label='All')
    if len(pareto_results) > 0:
        ax.scatter([r['inference_time'] for r in pareto_results], 
                  [r['iou'] for r in pareto_results], 
                  c='red', s=100, marker='*', label='Pareto-optimal')
    ax.set_xlabel('Inference Time (ms)')
    ax.set_ylabel('IoU')
    ax.set_title('IoU vs Speed')
    ax.legend()
    
    # F1 vs Parameters
    ax = axes[1, 0]
    ax.scatter(df['param_count'] / 1e6, df['f1_score'], alpha=0.5, label='All')
    if len(pareto_results) > 0:
        ax.scatter([r['param_count'] / 1e6 for r in pareto_results], 
                  [r['f1_score'] for r in pareto_results], 
                  c='red', s=100, marker='*', label='Pareto-optimal')
    ax.set_xlabel('Parameters (M)')
    ax.set_ylabel('F1 Score')
    ax.set_title('F1 vs Model Size')
    ax.legend()
    
    # Architecture comparison
    ax = axes[1, 1]
    architectures = df['architecture'].unique()
    for arch in architectures:
        arch_df = df[df['architecture'] == arch]
        ax.scatter(arch_df['inference_time'], arch_df['iou'], 
                  label=arch, alpha=0.7, s=60)
    ax.set_xlabel('Inference Time (ms)')
    ax.set_ylabel('IoU')
    ax.set_title('Architecture Comparison')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

## 3. Model Comparison by Architecture

In [ ]:
if 'df' in locals():
    # Group by architecture
    arch_summary = df.groupby('architecture').agg({
        'iou': ['mean', 'std', 'max'],
        'f1_score': ['mean', 'std', 'max'],
        'param_count': 'mean',
        'inference_time': 'mean'
    }).round(4)
    
    print("Performance by Architecture:")
    display(arch_summary)

In [ ]:
# Box plots
if 'df' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    df.boxplot(column='iou', by='architecture', ax=axes[0])
    axes[0].set_title('IoU Distribution by Architecture')
    axes[0].set_xlabel('Architecture')
    
    df.boxplot(column='inference_time', by='architecture', ax=axes[1])
    axes[1].set_title('Inference Time Distribution by Architecture')
    axes[1].set_xlabel('Architecture')
    
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

## 4. Encoder Comparison

In [ ]:
if 'df' in locals():
    encoder_summary = df.groupby('encoder').agg({
        'iou': ['mean', 'max'],
        'param_count': 'mean',
        'inference_time': 'mean'
    }).round(4)
    
    print("Performance by Encoder:")
    display(encoder_summary)

## 5. Training Curves (for individual experiments)

In [ ]:
# Load a specific experiment's training history
if latest_run:
    exp_dirs = [d for d in os.listdir(run_path) if os.path.isdir(os.path.join(run_path, d))]
    if exp_dirs:
        exp_path = os.path.join(run_path, exp_dirs[0])
        summary_path = os.path.join(exp_path, 'summary.json')
        
        if os.path.exists(summary_path):
            with open(summary_path) as f:
                exp_summary = json.load(f)
            
            if 'history' in exp_summary:
                history = exp_summary['history']
                
                fig, axes = plt.subplots(2, 2, figsize=(14, 10))
                
                # Loss
                ax = axes[0, 0]
                ax.plot(history['loss'], label='Train')
                ax.plot(history['val_loss'], label='Val')
                ax.set_title('Loss')
                ax.legend()
                ax.grid(True, alpha=0.3)
                
                # IoU
                ax = axes[0, 1]
                ax.plot(history.get('iou_score', []), label='Train')
                ax.plot(history.get('val_iou_score', []), label='Val')
                ax.set_title('IoU')
                ax.legend()
                ax.grid(True, alpha=0.3)
                
                # F1
                ax = axes[1, 0]
                ax.plot(history.get('dice_score', []), label='Train')
                ax.plot(history.get('val_dice_score', []), label='Val')
                ax.set_title('F1/Dice Score')
                ax.legend()
                ax.grid(True, alpha=0.3)
                
                # Learning Rate
                if 'lr' in history:
                    ax = axes[1, 1]
                    ax.plot(history['lr'])
                    ax.set_title('Learning Rate')
                    ax.grid(True, alpha=0.3)
                
                plt.tight_layout()
                plt.show()
                
                print(f"Experiment: {exp_summary.get('experiment_id', 'N/A')}")
                print(f"Final Val IoU: {exp_summary.get('final_val_iou', 0):.4f}")

## 6. Top Models Summary

In [ ]:
if 'df' in locals():
    # Top 10 by IoU
    top_iou = df.nlargest(10, 'iou')[['architecture', 'encoder', 'iou', 'f1_score', 
                                        'param_count', 'inference_time']]
    print("Top 10 Models by IoU:")
    display(top_iou)
    
    # Top 10 by F1
    top_f1 = df.nlargest(10, 'f1_score')[['architecture', 'encoder', 'iou', 'f1_score',
                                           'param_count', 'inference_time']]
    print("\nTop 10 Models by F1:")
    display(top_f1)
    
    # Best efficiency (IoU / inference_time)
    df['efficiency'] = df['iou'] / df['inference_time']
    top_efficiency = df.nlargest(10, 'efficiency')[['architecture', 'encoder', 'iou', 
                                                     'inference_time', 'efficiency']]
    print("\nTop 10 Models by Efficiency (IoU/ms):")
    display(top_efficiency)

## 7. Correlation Analysis

In [ ]:
if 'df' in locals():
    # Correlation matrix
    corr_cols = ['iou', 'f1_score', 'param_count', 'inference_time']
    corr_matrix = df[corr_cols].corr()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Correlation Matrix')
    plt.show()